In [4]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

TARGET_COLUMN = "WQI"             # Change to your target name, or use None if absent
VIF_THRESHOLD = 5.0               # 5 is a common investigation threshold; 10 is stricter
OUTPUT_FILE = "vif_results.csv"

df = pd.read_csv("Water Quality.csv")  

# Exclude the target from predictor features if it exists in the dataset
if TARGET_COLUMN and TARGET_COLUMN in df.columns:
    predictors = df.drop(columns=[TARGET_COLUMN])
else:
    predictors = df.copy()
    if TARGET_COLUMN:
        print(f"Note: target column '{TARGET_COLUMN}' was not found; using all columns as predictors.")

# VIF requires numeric data. Non-numeric columns need separate encoding before inclusion.
X = predictors.select_dtypes(include="number").copy()

# Replace infinite values and remove all-empty or constant columns
X = X.replace([float("inf"), float("-inf")], pd.NA)
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]

# Use complete rows so all VIF calculations use the same observations
X = X.dropna(axis=0, how="any")

if X.shape[1] < 2:
    raise ValueError("At least two varying numerical predictor columns are required.")
if len(X) <= X.shape[1] + 1:
    print("Warning: there are relatively few complete rows compared with the number of predictors.")

# Standardizing is not required for VIF. Add a constant for the intercept.
X_with_constant = add_constant(X, has_constant="add")
vif_rows = []

for i, column in enumerate(X_with_constant.columns):
    if column == "const":
        continue
    try:
        value = variance_inflation_factor(X_with_constant.values, i)
    except Exception:
        value = float("inf")
    vif_rows.append({"Feature": column, "VIF": value})

vif_df = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
vif_df.to_csv(OUTPUT_FILE, index=False)

print("VIF results (largest first):")
print(vif_df.to_string(index=False, formatters={"VIF": lambda x: f"{x:.3f}"}))
print(f"\nFull results saved as: {OUTPUT_FILE}")
print(f"\nFeatures with VIF >= {VIF_THRESHOLD}:")
high_vif = vif_df[vif_df["VIF"] >= VIF_THRESHOLD]
if high_vif.empty:
    print("No features crossed the selected threshold.")
else:
    print(high_vif.to_string(index=False, formatters={"VIF": lambda x: f"{x:.3f}"}))

print(
    "\nInterpretation: VIF near 1 usually means little multicollinearity; values "
    "above 5 deserve investigation, and values above 10 are often considered high. "
    "These are guidelines, not strict rules. Do not automatically delete a feature: "
    "check domain importance, correlation results, and model performance first."
)


Note: target column 'WQI' was not found; using all columns as predictors.
VIF results (largest first):
                    Feature   VIF
 MonitoringLocationLatitude 2.558
MonitoringLocationLongitude 2.553
   ResultAnalyticalMethodID 1.003
                ResultValue 1.001

Full results saved as: vif_results.csv

Features with VIF >= 5.0:
No features crossed the selected threshold.

Interpretation: VIF near 1 usually means little multicollinearity; values above 5 deserve investigation, and values above 10 are often considered high. These are guidelines, not strict rules. Do not automatically delete a feature: check domain importance, correlation results, and model performance first.
